# Deep Research End-to-End Workflow - May 2026 Release

## Overview
This notebook demonstrates the complete end-to-end workflow for deep research analysis, integrating multiple AI agents to provide comprehensive insights and recommendations.

## Solution Architecture

```mermaid
graph TD
    A[Raja - Anomaly<br/>COC_CMN_TRND_MTRC_INSGHT_STG<br/><br/>Top 3 States | Top 5 Providers | Top 5 DRG<br/>Top 5 Provider/DRG within each state]

    B[Rajib - Deep Dive<br/>COC_CMN_DATA_INSGHT_STG<br/>Deepdive & KeyInsight Summary<br/><br/>For each state: DRG, auth approvals, metrics]

    C[Semantic Model<br/>Claim, Member, Auth, Network, PI,<br/>Metric Definition]

    D[Correlation Agent<br/>Additional dimensions<br/>Product, Facility, Diagnosis, Procedure, Modifier<br/><br/>3 States | 5 DRG | 5 Providers]

    E[Pattern Analysis Agent<br/>Aggregates and finds common patterns<br/><br/>Pattern-1<br/>Pattern-N]

    F[Reimbursement Policy Agent - SME<br/><br/>Reimbursement Policy<br/>Reimbursement Policy]

    G[Recommendation Agent<br/>Aggregates all patterns and explanations<br/>to generate recommendations]

    H[Final Recommendations]

    A --> B
    B --> C
    B --> D
    D --> E
    E --> F
    E --> G
    F --> G
    G --> H

    style A fill:#e1f5ff,stroke:#00b8b8,stroke-width:2px,color:#173b7a
    style B fill:#e1f5ff,stroke:#00b8b8,stroke-width:2px,color:#173b7a
    style C fill:#e8eef9,stroke:#173b7a,stroke-width:2px,color:#173b7a
    style D fill:#ffffff,stroke:#00b8b8,stroke-width:2px,color:#173b7a
    style E fill:#fff4e1,stroke:#00b8b8,stroke-width:2px,color:#173b7a
    style F fill:#fff4e1,stroke:#00b8b8,stroke-width:2px,color:#173b7a
    style G fill:#e8f5e9,stroke:#00b8b8,stroke-width:2px,color:#173b7a
    style H fill:#e8f5e9,stroke:#173b7a,stroke-width:2px,color:#173b7a
```

## Workflow Steps

1. **Data Loading**: Load insights from Rajib's anomaly detection (KEY_INSIGHT and DEEP_DIVE)
2. **Pattern Analysis**: Extract actionable patterns from DEEP_DIVE reports
3. **Correlation Analysis**: Find additional dimensions (procedures, modifiers) driving each pattern
4. **Policy Extraction**: Retrieve relevant reimbursement policies for identified codes
5. **Recommendation Synthesis**: Aggregate findings to generate actionable recommendations

## API Endpoints
- Pattern Agent: `POST http://localhost:8000/agents/pattern_agent`
- Correlation Agent: `POST http://localhost:8000/agents/correlation`
- Reimbursement Policy Agent: `POST http://localhost:8000/agents/reimbursement_policy`
- Recommendation Agent: `POST http://localhost:8000/agents/recommendation_synthesis`

# Setup & Initialization

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
import copy
import pandas as pd
import json
import requests
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown, HTML

# Configure pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

print(f"Python: {sys.executable}")
print(f"Version: {sys.version}")

Python: /Users/AH45807/project/idiscovery-deep-research/.venv/bin/python
Version: 3.13.5 (main, Jun 11 2025, 15:36:57) [Clang 17.0.0 (clang-1700.0.13.3)]


In [3]:
# Load environment variables
project_root = Path.cwd().parent.parent.parent
env_path = project_root / ".env"
load_dotenv(env_path)

print(f"Project root: {project_root}")
print(f"Environment loaded: {env_path.exists()}")

Project root: /Users/AH45807/project/idiscovery-deep-research
Environment loaded: True


In [4]:
from deep_research_utils import SnowparkHelper, EHAPBase
from deep_research_utils.app_constant import AppConstants
from langchain_openai import ChatOpenAI

2026-05-28 15:05:40,765 - policy_extractor.system - INFO - === Policy Extractor Logging Initialized ===
2026-05-28 15:05:40,765 - policy_extractor.system - INFO - Log directory: /Users/AH45807/project/idiscovery-deep-research/notebooks/examples/2026-05/logs
2026-05-28 15:05:40,765 - policy_extractor.system - INFO - Max file size: 50.0MB
2026-05-28 15:05:40,766 - policy_extractor.system - INFO - Backup count: 10
2026-05-28 15:05:40,766 - policy_extractor.system - INFO - Console output enabled: True
2026-05-28 15:05:40,767 - policy_extractor.system - INFO - Console log level: INFO
2026-05-28 15:05:40,767 - policy_extractor.system - INFO - Console stream: stdout
2026-05-28 15:05:40,767 - policy_extractor.system - INFO - Process ID: 57383
2026-05-28 15:05:40,768 - policy_extractor.system - INFO - Component log levels:
2026-05-28 15:05:40,769 - policy_extractor.system - INFO -   policy_extractor.snowflake_store: WARNING
2026-05-28 15:05:40,769 - policy_extractor.system - INFO -   policy_ext

In [5]:
# Initialize Snowpark connection
snowpark_programmatic_connection_parameters = {
    "account": os.environ["SNOWFLAKE_ACCOUNT"],
    "user": os.environ["SNOWFLAKE_USER"],
    "password": os.environ["SNOWFLAKE_SECRET"],
    "warehouse": os.environ["SNOWFLAKE_WAREHOUSE"],
    "database": os.environ["SNOWFLAKE_DATABASE"],
    "schema": os.environ["SNOWFLAKE_SCHEMA"]
}

snowpark = SnowparkHelper(
    connection_type="programmatic",
    batch_size=10000,
    max_workers=6,
    enable_metrics=True,
    connection_pool_size=4,
    **snowpark_programmatic_connection_parameters
)

print("✓ Snowpark connection established")

2026-05-28 15:05:42,060 - deep_research_utils.snowflake_helper - INFO - 🔑 Configuring PROGRAMMATIC connection (password-based authentication)
2026-05-28 15:05:44,735 - deep_research_utils.snowflake_helper - INFO - Snowflake session created successfully
2026-05-28 15:05:52,568 - deep_research_utils.snowflake_helper - INFO - Initialized connection pool with 3 additional sessions
✓ Snowpark connection established


In [7]:
# Initialize EHAP authentication
EHAP = EHAPBase(
    base_url=os.environ.get("EHAP_BASE_URL"),
    client_id=os.environ.get("EHAP_CLIENT_ID"),
    client_secret=os.environ.get("EHAP_CLIENT_SECRET"),
    verify=os.environ.get("SSL_CERT_FILE")
)

print("✓ EHAP authentication initialized")

✓ EHAP authentication initialized


In [8]:
# Initialize LLM
from langchain_core.runnables import ConfigurableField
llm = ChatOpenAI(
    model=AppConstants.EHAP_LLM_MODEL,
    api_key=EHAP.get_token(),
    extra_body={
        "reasoning_effort": "medium",
        "summary": None
    }
).configurable_fields(
    openai_api_key=ConfigurableField(id="user_api_key")
)

llm_low_reasoning = ChatOpenAI(
    model=AppConstants.EHAP_LLM_MODEL,
    api_key=EHAP.get_token(),
    extra_body={
        "reasoning_effort": "low",
        "summary": None
    }
).configurable_fields(
    openai_api_key=ConfigurableField(id="user_api_key")
)

print("✓ LLM initialized")

2026-05-28 15:06:01,715 - deep_research_utils.ehap - INFO - Requesting new access token from https://api.horizon.elevancehealth.com/v2/oauth2/token with client_id: VOlF2INQArWlEG4icdjyI7js5ICMuwfM
2026-05-28 15:06:01,846 - deep_research_utils.ehap - INFO - Access token generated successfully.
**TOKEN** **TOKEN** **TOKEN** 


/Users/AH45807/project/idiscovery-deep-research/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.horizon.elevancehealth.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


✓ LLM initialized


In [9]:
# API Configuration
API_BASE_URL = "http://localhost:8000"
CONVERSATION_ID = f"tutorial_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

print(f"API Base URL: {API_BASE_URL}")
print(f"Conversation ID: {CONVERSATION_ID}")

API Base URL: http://localhost:8000
Conversation ID: tutorial_20260528_150602


# Step 1: Load Insights Data

Load the insights generated by Rajib's anomaly detection system. This includes both KEY_INSIGHT and DEEP_DIVE summaries.


This table is stored in Snowflake:
```sql
select * from U01_COC.COC_DTI_STG.coc_cmn_data_insght_stg
```

In [11]:
# Load insights data
# Note: Update this path to match your local environment
insights_csv_path = project_root / "notebooks/examples/2026-05/2026-05-18-UAT-data-insight-stg.csv"

# Alternative: You can also query directly from Snowflake if available
# df_insights = snowpark.session.table("COC_CMN_DATA_INSGHT_STG").to_pandas()

df_insights = pd.read_csv(insights_csv_path)
focus_hcc = "IP AUTH" # for 2026-05 release this is the HCC of interest
df_insights = df_insights[df_insights.STATSCL_MDL_CD == focus_hcc]
print(f"Loaded insights shape: {df_insights.shape}")
print(f"\nInsight types:")
print(df_insights.INSGHT_TYPE_NM.value_counts())
print(f"\nColumns: {list(df_insights.columns)}")

Loaded insights shape: (24, 18)

Insight types:
INSGHT_TYPE_NM
DEEP_DIVE      12
KEY_INSIGHT    12
Name: count, dtype: int64

Columns: ['EDL_LOAD_DTM', 'EDL_RUN_ID', 'EDL_SOR_CD', 'KF_TMS', 'EDL_SCRTY_LVL_CD', 'EDL_LOB_CD', 'EDL_EXTRNL_LOAD_CD', 'EDL_CREAT_DTM', 'EDL_INCRMNTL_LOAD_DTM', 'SNAP_YEAR_MNTH_NBR', 'TRND_TM_PRD_END_MNTH_NBR', 'TRND_TM_PRD_CD', 'LOB_CD', 'LOB_SHRT_DESC', 'STATSCL_MDL_CD', 'INSGHT_TYPE_NM', 'JSON_TXT', 'OFSHR_EXCLSN_SOR_CD']


# Step 2: Process First Row - End-to-End Example

We'll walk through the complete workflow for the first row to demonstrate each agent interaction.

In [12]:
for snap_year_mnth_nbr in df_insights.SNAP_YEAR_MNTH_NBR.unique():
    for trnd_tm_prd_end_mnth_nbr in df_insights.TRND_TM_PRD_END_MNTH_NBR.unique():
        for trnd_tm_prd_cd in df_insights.TRND_TM_PRD_CD.unique():
            for lob_shrt_desc in df_insights.LOB_SHRT_DESC.unique():
                for statscl_mdl_cd in df_insights.STATSCL_MDL_CD.unique():
                    print(f"{snap_year_mnth_nbr}, {trnd_tm_prd_end_mnth_nbr}, {trnd_tm_prd_cd}, {lob_shrt_desc}, {statscl_mdl_cd}")
                    break
                break
            break
        break
    break

202604, 202601, R3, Commercial, IP AUTH


In [13]:
first_anomaly = df_insights[(df_insights.SNAP_YEAR_MNTH_NBR == snap_year_mnth_nbr) &
                              (df_insights.TRND_TM_PRD_END_MNTH_NBR == trnd_tm_prd_end_mnth_nbr) &
                              (df_insights.TRND_TM_PRD_CD == trnd_tm_prd_cd) &
                              (df_insights.LOB_SHRT_DESC == lob_shrt_desc) & 
                              (df_insights.STATSCL_MDL_CD == statscl_mdl_cd) &
                              (df_insights.INSGHT_TYPE_NM == 'KEY_INSIGHT')]
anomaly_json = json.loads(json.loads(first_anomaly.JSON_TXT.iloc[0]))
anomaly_json

{'whats_happening': '',
 'top_contributors': {'provider_trends': [{'name': 'EMORY HILLANDALE HOSPITAL',
    'insight': 'Experienced a substantial increase in authorizations',
    'percentage_change': '+490%'},
   {'name': 'EMORY DECATUR HOSPITAL',
    'insight': 'Showed a significant rise in authorization counts',
    'percentage_change': '+239%'},
   {'name': 'YALE NEW HAVEN HOSPITAL',
    'insight': 'Noted a considerable increase in authorizations',
    'percentage_change': '+34%'},
   {'name': "HENRICO DOCTORS' HOSPITAL",
    'insight': 'Saw a moderate increase in authorization counts',
    'percentage_change': '+22%'}],
  'states': [{'name': 'ME',
    'insight': 'Had a significant increase in authorization counts',
    'percentage_change': '+52%'},
   {'name': 'CO',
    'insight': 'Experienced a notable rise in authorization counts',
    'percentage_change': '+14%'}],
  'drgs': [{'name': 'Ungroupable',
    'insight': 'Showed a substantial increase in authorizations',
    'percentag

In [14]:
first_deep_dive = df_insights[(df_insights.SNAP_YEAR_MNTH_NBR == snap_year_mnth_nbr) &
                              (df_insights.TRND_TM_PRD_END_MNTH_NBR == trnd_tm_prd_end_mnth_nbr) &
                              (df_insights.TRND_TM_PRD_CD == trnd_tm_prd_cd) &
                              (df_insights.LOB_SHRT_DESC == lob_shrt_desc) & 
                              (df_insights.STATSCL_MDL_CD == statscl_mdl_cd) &
                              (df_insights.INSGHT_TYPE_NM == 'DEEP_DIVE')]

deep_dive_json = json.loads(json.loads(first_deep_dive.JSON_TXT.iloc[0]))
deep_dive_json

{'report_title': 'IP Authorization Insights - R3 (Snap Month: 202604, Period End: 202601) - Commercial',
 'national_summary': {'description': 'The national total for Commercial IP authorizations R3 is 33,810 (AMFINR). This report covers the top variance drivers by State, DRG, and Provider for the AUTH_CNT metric.'},
 'top_state_drivers': {'section_title': 'TOP STATE DRIVERS',
  'states': [{'state_name': 'CT',
    'overview': 'CT has 1,802 auths out of 33,810 nationally (5.33% of the national total).',
    'medical_necessity_review_mix': 'In CT, of 1,802 authorizations, 95.45% require Medical Necessity review.',
    'service_driver': 'In CT, of total 1,802 authorizations, IP Med/Surg contributes 1,124 authorizations, accounting for 62.38%; IP BH contributes 511 authorizations, accounting for 28.36%; IP OB Dlvry NB contributes 115 authorizations, accounting for 6.38%; NF contributes 55 authorizations, accounting for 3.05%.',
    'authorization_status_mix': 'In CT, of the 1,802 authorizat

## Step 2.1: Correlation Agent/ Waterfall Agent

The Correlation Agent identifies additional dimensions (procedures, modifiers, etc.) that drive the observed patterns. It drill down to find the most significant factors contributing to the pattern.

Start API server:
```bash
source .venv/bin/activate && uvicorn packages.agents.src.agent_api:app --
reload --host 0.0.0.0 --port 8000
```

In [15]:
from datetime import datetime

def get_ecap_start_month(trnd_tm_prd_cd: str, trnd_tm_prd_end_mnth_nbr: int) -> int:
    """
    Compute start month (YYYYMM) for a given ECAP time period.

    Args:
        trnd_tm_prd_cd (int): One of ["R3", "R6", "R12", "YTD"]
        trnd_tm_prd_end_mnth_nbr (int): End month in YYYYMM format

    Returns:
        int: Start month in YYYYMM format
    """

    end_date = datetime.strptime(str(trnd_tm_prd_end_mnth_nbr), "%Y%m")

    def subtract_months(dt, months):
        year = dt.year
        month = dt.month - months

        while month <= 0:
            month += 12
            year -= 1

        return datetime(year, month, 1)

    if trnd_tm_prd_cd.startswith("R"):
        months = int(trnd_tm_prd_cd[1:])
        # standard rolling window (inclusive)
        start_date = subtract_months(end_date, months - 1)

    elif trnd_tm_prd_cd == "YTD":
        start_date = datetime(end_date.year, 1, 1)

    else:
        raise ValueError(f"Unsupported trnd_tm_prd_cd: {trnd_tm_prd_cd}")

    return int(start_date.strftime("%Y%m"))

# Test cases
print(get_ecap_start_month("R3", 202501))   # Expected: 202411
print(get_ecap_start_month("R6", 202501))   # Expected: 202408
print(get_ecap_start_month("R12", 202501))  # Expected: 202402
print(get_ecap_start_month("YTD", 202512))  # Expected: 202501

202411
202408
202402
202501


In [16]:
def convert_current_ecap_time_to_previous_year(current_period_start: int,
                                               current_period_end: int) -> tuple[int, int]:
    """
    Convert ECAP period (YYYYMM) to previous year period.

    Args:
        current_period_start (int): Start period in YYYYMM format
        current_period_end (int): End period in YYYYMM format

    Returns:
        tuple[int, int]: (previous_period_start, previous_period_end)
    """

    def shift_to_previous_year(period: int) -> int:
        year = period // 100
        month = period % 100

        if not (1 <= month <= 12):
            raise ValueError(f"Invalid month in period: {period}")

        return (year - 1) * 100 + month

    previous_period_start = shift_to_previous_year(current_period_start)
    previous_period_end = shift_to_previous_year(current_period_end)

    return previous_period_start, previous_period_end

In [17]:
# if you dont set them here, it will pick up the defaults from the configs/correlation_pattern/coc_ecap_ip_auth_sematic_view_with_samples.yaml
current_time_period_end = int(trnd_tm_prd_end_mnth_nbr) # critical step to convert the numpy to int64, else API will throw error
current_time_period_start = get_ecap_start_month(trnd_tm_prd_cd, current_time_period_end)
# print("Current period:", current_time_period_start, current_time_period_end)

previous_period_start, previous_period_end = convert_current_ecap_time_to_previous_year(current_time_period_start, current_time_period_end)
# print(f"Previous period: {previous_period_start} to {previous_period_end}")
conversation_id = f"tutorial-{statscl_mdl_cd}-{lob_shrt_desc}-{snap_year_mnth_nbr}-{trnd_tm_prd_cd}-{trnd_tm_prd_end_mnth_nbr}".replace(" ", "_")

correlation_agent_common_payload = {
    "conversation_id": conversation_id,
    "context":{
      "analysis_mode_parameters": {
        "drill_metric": ["expense_detail.total_paid"],
        "period": {
          "rolling_time_dimension": "expense_detail.incurred_month",
          "current_period": {
            "start_time": current_time_period_start,
            "end_time": current_time_period_end
          },
          "previous_period": {
            "start_time": previous_period_start,
            "end_time": previous_period_end
          }
        }
      },
      "filters": [
        {
          "field": "snap_month",
          "operator": "=",
          "value": int(snap_year_mnth_nbr),
          "source": "dimension_match"
        },
        {
          "field": "lob_description",
          "operator": "=",
          "value": lob_shrt_desc,
          "source": "dimension_match"
        }
      ]
    },

  }
# add the HCC filter
if statscl_mdl_cd == "IP AUTH":
  correlation_agent_common_payload["context"]["filters"].append({
        "field": "hcc_high",
        "operator": "=",
        "value": "IP",
        "source": "dimension_match"
      })
else:
    correlation_agent_common_payload["context"]["filters"].append({
        "field": "hcc_medium",
        "operator": "=",
        "value": statscl_mdl_cd,
        "source": "dimension_match"
    })
# correlation_agent_common_payload

In [ ]:
%%time
# Call Correlation Agent for all the states 
correlation_agent_url = f"{API_BASE_URL}/agents/correlation"
correlation_results = {
    "states": {},
    "providers": {},
    "drgs": {}
}
for state in anomaly_json["top_contributors"]["states"]:
    correlation_agent_payload = copy.deepcopy(correlation_agent_common_payload)
    correlation_agent_payload["query"] = f"Where did change happen for state {state['name']}? It {state['insight'].lower()} by {state['percentage_change']}"
    correlation_agent_payload["context"]["filters"].append({
        "field": "service_area_state",
        "operator": "=",
        "value": state["name"],
        "source": "dimension_match"
    })
    print(f"Calling Correlation Agent for state '{state['name']}'...")
    correlation_response = requests.post(correlation_agent_url, json=correlation_agent_payload)
    correlation_result = correlation_response.json()

    print(f"Status: {correlation_response.status_code}")
    print(f"Success: {correlation_result.get('status', False)}")
    correlation_results["states"][state["name"]] = correlation_result

# run for all provider_trends
for provider in anomaly_json["top_contributors"]["provider_trends"]:
    correlation_agent_payload = copy.deepcopy(correlation_agent_common_payload)
    correlation_agent_payload["query"] = f"Where did change happen for provider {provider['name']}? It {provider['insight'].lower()} by {provider['percentage_change']}"
    correlation_agent_payload["context"]["filters"].append({
        "field": "rendering_provider_name",
        "operator": "=",
        "value": provider["name"],
        "source": "dimension_match"
    })
    print(f"Calling Correlation Agent for provider '{provider['name']}'...")
    correlation_response = requests.post(correlation_agent_url, json=correlation_agent_payload)
    correlation_result = correlation_response.json()

    print(f"Status: {correlation_response.status_code}")
    print(f"Success: {correlation_result.get('status', False)}")
    correlation_results["providers"][provider["name"]] = correlation_result


#  run for all drgs
for drg in anomaly_json["top_contributors"]["drgs"]:
    correlation_agent_payload = copy.deepcopy(correlation_agent_common_payload)
    correlation_agent_payload["query"] = f"Where did change happen for drg {drg['name']}? It {drg['insight'].lower()} by {drg['percentage_change']}"
    correlation_agent_payload["context"]["filters"].append({
        "field": "drg_name",
        "operator": "=",
        "value": drg["name"],
        "source": "dimension_match"
    })
    print(f"Calling Correlation Agent for drg '{drg['name']}'...")
    correlation_response = requests.post(correlation_agent_url, json=correlation_agent_payload)
    correlation_result = correlation_response.json()

    print(f"Status: {correlation_response.status_code}")
    print(f"Success: {correlation_result.get('status', False)}")
    correlation_results["drgs"][drg["name"]] = correlation_result

Calling Correlation Agent for state 'ME'...
Status: 200
Success: success
Calling Correlation Agent for state 'CO'...
Status: 200
Success: success
Calling Correlation Agent for provider 'EMORY HILLANDALE HOSPITAL'...
Status: 200
Success: success
Calling Correlation Agent for provider 'EMORY DECATUR HOSPITAL'...
Status: 200
Success: success
Calling Correlation Agent for provider 'YALE NEW HAVEN HOSPITAL'...
Status: 200
Success: success
Calling Correlation Agent for provider 'HENRICO DOCTORS' HOSPITAL'...
Status: 200
Success: success
Calling Correlation Agent for drg 'Ungroupable'...
Status: 200
Success: success
Calling Correlation Agent for drg 'Chemotherapy without Acute Leukemia as Secondary Diagnosis with MCC'...
Status: 200
Success: success
Calling Correlation Agent for drg 'Tendonitis, Myositis and Bursitis without MCC'...
Status: 200
Success: success
Calling Correlation Agent for drg 'Cesarean Section without Sterilization without CC/MCC'...
Status: 200
Success: success
Calling Cor

In [18]:
# store this for use in future cells
CONVERSATION_ID = conversation_id

In [ ]:
# fname = f"anomaly{datetime.now().strftime('%Y%m%d')}.json"
# print(fname)
# with open(fname, 'w') as f:
#     json.dump(anomaly_json, f, indent=4)  # indent makes it human-readable

In [ ]:
fname = f"correlation_results_{datetime.now().strftime('%Y%m%d')}_{conversation_id.replace("tutorial-", "")}.json"
print(fname)
with open(fname, 'w') as f:
    json.dump(correlation_results, f, indent=4)  # indent makes it human-readable
    
# with open("correlation_results_20260514_IP_AUTH-Commercial-202604-R3-202601.json", 'r') as f:
#     correlation_results = json.load(f)

## Step 2.2: Pattern Analysis Agent

The Pattern Analysis Agent extracts meaningful patterns from the DEEP_DIVE report, identifying key trends, outliers, and actionable insights.


In [ ]:
# Call Pattern Analysis Agent
pattern_agent_url = f"{API_BASE_URL}/agents/pattern_agent"
pattern_request = {
    "conversation_id": CONVERSATION_ID,
    "query": "Summarize the highest-impact authorization and provider mix themes from the completed correlation analysis.",
    "context": {
        "anomaly_context" : anomaly_json,
        "deep_dive_report": deep_dive_json,
        "correlation_results": correlation_results
    }
}

print("Calling Pattern Analysis Agent...")
pattern_response = requests.post(pattern_agent_url, json=pattern_request)
pattern_results = pattern_response.json()

print(f"Status: {pattern_response.status_code}")
print(f"Success: {pattern_results.get('status', False)}")

Calling Pattern Analysis Agent...
Status: 200
Success: success


In [82]:
# print(json.dumps(pattern_result, indent=2))

In [ ]:
fname = f"pattern_results_{datetime.now().strftime('%Y%m%d')}_{conversation_id.replace('tutorial-', '')}.json"
print(fname)
with open(fname, 'w') as f:
    json.dump(pattern_results, f, indent=4)  # indent makes it human-readable
    
    
# with open("pattern_results_20260527_IP_AUTH-Commercial-202604-R3-202601.json", 'r') as f:
#     pattern_results = json.load(f)

## Step 3: Reimbursement Agent

The Reimbursement Agent analyzes payer policies to understand reimbursement rules, bundling logic, and denial conditions for identified DRG codes. This provides context for understanding cost drivers and potential interventions.

In [20]:
# Load pattern results
import json
 
with open('pattern_results_20260527_IP_AUTH-Commercial-202604-R3-202601.json', 'r') as f:
    pattern_results = json.load(f)
 
# Extract pattern 5 and cards
patterns = pattern_results['output']['business_patterns']
pattern_5 = next(p for p in patterns if p['pattern_rank'] == 5)
 
# Get all cards (needed for LOB/Product extraction)
all_cards = pattern_results['output'].get('cards', [])
 
# Create the complete input payload
payload = {
    "context": {
        "pattern": pattern_5,
        "cards": all_cards  # ← CRITICAL for LOB/Product extraction!
    },
    "conversation_id": "test-pattern-5-cesarean",
    "query": "Analyze reimbursement policies for pattern 5: California cesarean delivery",
    "job_id": "test-pattern-5-20260528"
}

In [42]:
print(json.dumps(payload, indent=2))

{
  "context": {
    "pattern": {
      "pattern_rank": 5,
      "top_pattern": "California cesarean delivery spend is rising through more complex OB mix",
      "pattern_type": "clinical_case_mix",
      "what_is_impacting": "Commercial maternity / Inpatient OB delivery / Cesarean section",
      "priority_entities": {
        "states": [
          "CA"
        ],
        "providers": [
          "UCHEALTH MEMORIAL HOSPITAL CENTRAL"
        ],
        "products": [],
        "facility_types": [
          "ACUTE HOSPITAL"
        ],
        "clinical_categories": [
          "IP OB Dlvry/Well NB"
        ]
      },
      "key_driver_codes": [
        "Cesarean Section without Sterilization without CC/MCC",
        "PRE-EXISTING HTN PRE-ECLAMP COMP CB",
        "POST-TERM PREGNANCY",
        "MAT CARE LW TRANS SCAR PREV C/S DEL",
        "TWIN PG CHORIONIC/MONOAMNIOT 2ND TM"
      ],
      "impact_summary": {
        "primary_metric": "total_paid",
        "direction": "increase",
     

In [36]:
# Call Reimbursment Agent
reimbursement_agent_url = f"{API_BASE_URL}/agents/reimbursement_policy"
reimbursement_request = payload

print("Calling Reimbursment Agent...")
reimbursement_response = requests.post(reimbursement_agent_url, json=reimbursement_request)
reimbursement_results = reimbursement_response.json()

print(f"Status: {reimbursement_response.status_code}")
print(f"Success: {reimbursement_results.get('status', False)}")

Calling Reimbursment Agent...
Status: 200
Success: success


In [37]:
reimbursement_results

{'job_id': 'test-pattern-5-20260528',
 'conversation_id': 'test-pattern-5-cesarean',
 'agent': 'reimbursement_policy',
 'status': 'success',
 'output': {'pattern_rank': 5,
  'summary_table': {'title': 'Payer Policy Summary',
   'subtitle': 'California cesarean delivery spend is rising through more complex OB mix',
   'columns': [{'id': 'payer_org',
     'label': 'Payer Organization',
     'type': 'text'},
    {'id': 'maternity_admission_auth',
     'label': 'Maternity Admit Auth',
     'type': 'text'},
    {'id': 'multiple_birth_bundling',
     'label': 'Multiple Birth Rules',
     'type': 'text'},
    {'id': 'appeals_process',
     'label': 'Appeals Process\n(Documented)',
     'type': 'badge'},
    {'id': 'policy_effective_date',
     'label': 'Policy Effective Date\n(Last Updated)',
     'type': 'date'}],
   'rows': [{'payer_org': 'Cigna',
     'appeals_process': '-',
     'policy_effective_date': 'N/A',
     'maternity_admission_auth': 'No maternity admit authorization requirement 

In [ ]:
fname = f"reimbursement_results_{datetime.now().strftime('%Y%m%d')}_{conversation_id.replace("tutorial-", "")}.json"
print(fname)
with open(fname, 'w') as f:
    json.dump(reimbursement_results, f, indent=4)  # indent makes it human-readable
    
# with open("correlation_results_20260514_IP_AUTH-Commercial-202604-R3-202601.json", 'r') as f:
#     correlation_results = json.load(f)

## Step 4: Recommendation Agent

The Recommendation Agent synthesizes insights from pattern analysis and reimbursement policies to generate actionable recommendations. It uses decision tree rules and UI-optimized prompts to ensure concise, specific output.

In [ ]:
# Recommendation Agent Configuration
from deep_research_agents.decision_tree_rules import DecisionTreeRuleEngine

# Decision tree rules path
DECISION_TREE_PATH = project_root / "configs" / "decision_tree_rules.yaml"

# UI Display Constraints (Word Limits)
MAX_DESCRIPTION_WORDS = 100
MAX_EVIDENCE_WORDS = 15
MAX_STORY_ALIGNMENT_WORDS = 30
MAX_PEER_BENCHMARKING_WORDS = 25

# Validation Settings
REQUIRE_SPECIFICITY = True
SPECIFICITY_THRESHOLD = 0.75
MAX_RETRY_ATTEMPTS = 2

print("Configuration:")
print(f"  Decision Tree: {DECISION_TREE_PATH}")
print(f"  Max Description: {MAX_DESCRIPTION_WORDS} words")
print(f"  Max Evidence: {MAX_EVIDENCE_WORDS} words")
print(f"  Specificity Required: {REQUIRE_SPECIFICITY}")

In [ ]:
# Load decision tree rules
if DECISION_TREE_PATH.exists():
    try:
        rule_engine = DecisionTreeRuleEngine(str(DECISION_TREE_PATH))
        rule_count = rule_engine.get_total_rule_count()
        category_count = len(rule_engine.get_category_names())
        print(f"✓ Decision tree rules loaded")
        print(f"  Total rules: {rule_count}")
        print(f"  Categories: {category_count}")
        
        # Format rules for LLM
        rules_text = rule_engine.format_rules_compact()
        print(f"  Formatted for LLM: {len(rules_text)} characters")
    except Exception as e:
        print(f"❌ Failed to load decision tree: {e}")
        rule_engine = None
        rules_text = ""
else:
    print(f"⚠ Decision tree YAML not found at: {DECISION_TREE_PATH}")
    rule_engine = None
    rules_text = ""

In [ ]:
# Prepare input data for recommendation agent
# Following experimental notebook approach: pattern-centric structure
# Each pattern gets its own entry with embedded explanation data

print("[COMBINING DATA] Building unified input from loaded sources...")
print("="*80)

input_data = []

if pattern_results and 'output' in pattern_results:
    business_patterns = pattern_results['output'].get('business_patterns', [])
    cards = pattern_results['output'].get('cards', [])
    
    # Process each business pattern
    for pattern in business_patterns:
        pattern_rank = pattern.get('pattern_rank', 0)
        pattern_title = pattern.get('top_pattern', '')
        pattern_type = pattern.get('pattern_type', '')
        service_category = pattern.get('what_is_impacting', '')
        
        # Extract evidence from pattern
        evidence_summary = pattern.get('evidence_summary', [])
        impact_summary = pattern.get('impact_summary', {})
        
        # Build claim evidence section
        claim_evidence = {
            "summary": pattern.get('pattern_details', ''),
            "details": evidence_summary,
            "impact": impact_summary.get('estimated_delta', 'N/A'),
            "direction": impact_summary.get('direction', 'unknown')
        }
        
        # Build pattern-specific reimbursement section
        pattern_payor_summaries = [
            s for s in payor_pattern_summary 
            if s.get('pattern_rank') == pattern_rank
        ]
        
        reimbursement_section = {
            "pattern_specific": True,
            "payor_summaries": pattern_payor_summaries,
            "policies_analyzed": len([
                r for r in results_reimb 
                if r is not None and r.get('pattern_rank') == pattern_rank
            ])
        }
        
        # Build unified pattern entry
        unified_entry = {
            "rank": pattern_rank,
            "pattern_title": pattern_title,
            "pattern_type": pattern_type,
            "pattern_description": pattern.get('pattern_details', ''),
            "service_category": service_category,
            "priority_entities": pattern.get('priority_entities', {}),
            "key_driver_codes": pattern.get('key_driver_codes', []),
            "why_it_matters": pattern.get('why_it_matters', ''),
            "recommended_next_step": pattern.get('recommended_next_step', ''),
            "validation_needed": pattern.get('validation_needed', False),
            "downstream_routes": pattern.get('downstream_routes', []),
            "explanation": {
                "claim_evidence": claim_evidence,
                "reimbursement": reimbursement_section
                # NO correlation per requirements
            }
        }
        
        input_data.append(unified_entry)
    
    print(f"✓ Combined input data prepared: {len(input_data)} pattern(s)")
    
    if input_data:
        print(f"\nPattern 1: {input_data[0]['pattern_title']}")
        print(f"  Service Category: {input_data[0]['service_category']}")
        has_reimb = 'reimbursement' in input_data[0].get('explanation', {})
        has_corr = 'correlation' in input_data[0].get('explanation', {})
        print(f"  Has Reimbursement Data: {has_reimb}")
        print(f"  Has Correlation Data: {has_corr}")
        print(f"  Payor Summaries: {len(input_data[0]['explanation']['reimbursement']['payor_summaries'])}")
        
        if len(input_data) > 1:
            print(f"\nPattern 2: {input_data[1]['pattern_title']}")
            print(f"  Service Category: {input_data[1]['service_category']}")
            print(f"  Payor Summaries: {len(input_data[1]['explanation']['reimbursement']['payor_summaries'])}")
else:
    print("❌ No pattern data available to combine")

# Serialize for LLM
input_json_str = json.dumps(input_data, indent=2, ensure_ascii=False)

print("="*80)
print(f"✓ Input data prepared for recommendation agent")
print(f"  Total patterns: {len(input_data)}")
print(f"  Total input size: {len(input_json_str)} characters")
print(f"  Structure: Pattern-centric with per-pattern reimbursement data")
print(f"  Correlation included: ✗ (excluded per requirements)")

In [ ]:
# Define recommendation prompts
SYSTEM_PROMPT_RECOMM = f"""You are a healthcare cost management expert generating actionable recommendations.

# Decision Tree Rules
{rules_text}

# Your Task
Generate clear, specific, actionable recommendations based on pattern analysis and reimbursement policy findings.

# Output Requirements
- **Specificity**: Use concrete identifiers (DRG codes, provider names, dollar amounts, percentages)
- **Brevity**: Respect word count limits strictly
  - Description: MAX {MAX_DESCRIPTION_WORDS} words
  - Evidence: MAX {MAX_EVIDENCE_WORDS} words per bullet
  - Story Alignment: MAX {MAX_STORY_ALIGNMENT_WORDS} words per bullet
- **Story Alignment**: Explain WHY each recommendation matters using decision tree context
- **Actionability**: Focus on what can be done, not just what was observed

Return ONLY valid JSON matching the schema below."""

print("✓ System prompt defined")
print(f"  Includes {len(rules_text)} characters of decision tree rules")

In [ ]:
# Define user prompt
USER_PROMPT_RECOMM = f"""# Input Data

{input_json_str}

# Task
Analyze the pattern insights and reimbursement findings to generate 3-5 prioritized recommendations.

# Output Schema
```json
{{
  "recommendations": [
    {{
      "rank": 1,
      "title": "Concise recommendation title (MAX 10 words)",
      "description": "Detailed description (MAX {MAX_DESCRIPTION_WORDS} words)",
      "priority": "high | medium | low",
      "category": "policy | provider | utilization | coding | other",
      "evidence": [
        "Specific evidence point (MAX {MAX_EVIDENCE_WORDS} words)"
      ],
      "story_alignment": [
        "Why this matters based on decision tree (MAX {MAX_STORY_ALIGNMENT_WORDS} words)"
      ],
      "estimated_impact": "Dollar amount or percentage if quantifiable",
      "next_steps": [
        "Concrete action item"
      ]
    }}
  ]
}}
```

# Validation Rules
- Each recommendation must have at least 2 evidence points
- Each recommendation must have at least 1 story alignment point
- Use specific identifiers (DRG codes, provider names, amounts)
- Respect all word count limits strictly
- Return ONLY valid JSON, no markdown"""

print("✓ User prompt defined")
print(f"  Input data size: {len(input_json_str)} characters")

In [ ]:
# Invoke LLM for recommendation generation
print("[LLM] Generating recommendations...")
print("="*80)

recommendations_result = None
attempt_count = 0

while attempt_count < MAX_RETRY_ATTEMPTS and recommendations_result is None:
    attempt_count += 1
    print(f"\nAttempt {attempt_count}/{MAX_RETRY_ATTEMPTS}")
    
    try:
        messages_recomm = [
            {"role": "system", "content": SYSTEM_PROMPT_RECOMM},
            {"role": "user", "content": USER_PROMPT_RECOMM}
        ]
        
        # Invoke LLM with dynamic token
        llm_with_dynamic_key = llm_no_reasoning.with_config(
            configurable={"user_api_key": EHAP.get_token()}
        )
        response_recomm = llm_with_dynamic_key.invoke(messages_recomm)
        content_recomm = response_recomm.content.strip()
        
        # Remove markdown code fences if present
        if content_recomm.startswith("```"):
            lines = content_recomm.split("\n")
            if len(lines) > 2:
                content_recomm = "\n".join(lines[1:-1])
        
        # Parse JSON
        recommendations_result = json.loads(content_recomm)
        
        print(f"✓ LLM generated {len(recommendations_result.get('recommendations', []))} recommendations")
        
    except Exception as e:
        print(f"✗ Attempt {attempt_count} failed: {e}")
        if attempt_count >= MAX_RETRY_ATTEMPTS:
            print("❌ Max retry attempts reached")
        else:
            print("  Retrying...")

if recommendations_result:
    print(f"\n{'='*80}")
    print(f"✓ Recommendations generation complete")
else:
    print(f"\n{'='*80}")
    print(f"❌ Failed to generate recommendations")

In [ ]:
# Validate recommendations output
if recommendations_result:
    print("="*80)
    print("VALIDATION REPORT")
    print("="*80)
    
    recommendations = recommendations_result.get('recommendations', [])
    
    for i, rec in enumerate(recommendations, 1):
        print(f"\nRecommendation {i}: {rec.get('title', 'N/A')}")
        print(f"  Priority: {rec.get('priority', 'N/A')}")
        print(f"  Category: {rec.get('category', 'N/A')}")
        
        # Check word counts
        desc = rec.get('description', '')
        desc_words = len(desc.split())
        print(f"  Description: {desc_words} words (limit: {MAX_DESCRIPTION_WORDS})")
        if desc_words > MAX_DESCRIPTION_WORDS:
            print(f"    ⚠ EXCEEDS LIMIT")
        
        # Check evidence
        evidence = rec.get('evidence', [])
        print(f"  Evidence points: {len(evidence)}")
        for j, ev in enumerate(evidence, 1):
            ev_words = len(ev.split())
            status = "✓" if ev_words <= MAX_EVIDENCE_WORDS else "⚠"
            print(f"    {status} Evidence {j}: {ev_words} words (limit: {MAX_EVIDENCE_WORDS})")
        
        # Check story alignment
        story_align = rec.get('story_alignment', [])
        print(f"  Story alignment points: {len(story_align)}")
        for j, sa in enumerate(story_align, 1):
            sa_words = len(sa.split())
            status = "✓" if sa_words <= MAX_STORY_ALIGNMENT_WORDS else "⚠"
            print(f"    {status} Alignment {j}: {sa_words} words (limit: {MAX_STORY_ALIGNMENT_WORDS})")
    
    print(f"\n{'='*80}")
    print(f"Total recommendations: {len(recommendations)}")
else:
    print("No recommendations to validate")

In [ ]:
# Export recommendations
if recommendations_result:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_file_recomm = f"recommendations_{statscl_mdl_cd}_{lob_shrt_desc}_{snap_year_mnth_nbr}_{timestamp}.json".replace(" ", "_")
    
    # Add metadata to output
    export_data_recomm = {
        "metadata": {
            "snap_month": int(snap_year_mnth_nbr),
            "period_end": int(trnd_tm_prd_end_mnth_nbr),
            "period_code": trnd_tm_prd_cd,
            "lob": lob_shrt_desc,
            "statistical_model": statscl_mdl_cd,
            "conversation_id": conversation_id
        },
        "generation_timestamp": timestamp,
        "decision_tree_rules_used": rule_count if rule_engine else 0,
        "input_sources": {
            "pattern_analysis": True,
            "reimbursement_analysis": True,
            "correlation_analysis": False
        },
        "total_patterns_analyzed": len(input_data) if input_data else 0,
        "recommendations": recommendations_result
    }
    
    with open(output_file_recomm, 'w') as f:
        json.dump(export_data_recomm, f, indent=2)
    
    print(f"✓ Recommendations saved to: {output_file_recomm}")
    print(f"  Total patterns: {len(input_data) if input_data else 0}")
    print(f"  Total recommendations: {len(recommendations_result.get('recommendations', []))}") 
    print(f"  Input sources: Pattern ✓, Reimbursement ✓, Correlation ✗")
else:
    print("No recommendations to export")

In [ ]:
# Display recommendations summary
if recommendations_result:
    print("="*80)
    print("RECOMMENDATIONS SUMMARY")
    print("="*80)
    
    recommendations = recommendations_result.get('recommendations', [])
    
    for i, rec in enumerate(recommendations, 1):
        print(f"\n{'='*80}")
        print(f"Recommendation {i} [{rec.get('priority', 'N/A').upper()}]")
        print(f"{'='*80}")
        print(f"\n**Title:** {rec.get('title', 'N/A')}")
        print(f"**Category:** {rec.get('category', 'N/A')}")
        print(f"**Impact:** {rec.get('estimated_impact', 'Not quantified')}")
        
        print(f"\n**Description:**")
        print(f"{rec.get('description', 'N/A')}")
        
        print(f"\n**Evidence:**")
        for j, ev in enumerate(rec.get('evidence', []), 1):
            print(f"  {j}. {ev}")
        
        print(f"\n**Why This Matters:**")
        for j, sa in enumerate(rec.get('story_alignment', []), 1):
            print(f"  {j}. {sa}")
        
        print(f"\n**Next Steps:**")
        for j, ns in enumerate(rec.get('next_steps', []), 1):
            print(f"  {j}. {ns}")
    
    print(f"\n{'='*80}")
else:
    print("No recommendations available")

## Summary

This notebook demonstrates the complete end-to-end workflow for deep research analysis:

1. **✓ Data Loading**: Loaded insights from Rajib's anomaly detection (KEY_INSIGHT and DEEP_DIVE)
2. **✓ Correlation Analysis**: Identified additional dimensions (procedures, modifiers) driving patterns
3. **✓ Reimbursement Analysis**: Analyzed payer policies for DRG codes
4. **✓ Recommendation Synthesis**: Generated actionable recommendations combining pattern insights and reimbursement findings

### Key Outputs

- **Correlation Results**: Stored in correlation workflow cells
- **Reimbursement Results**: `reimbursement_agent_payor_summary_*.json`
- **Recommendations**: `recommendations_*.json`

### Data Sources Used for Recommendations

- ✓ Pattern Analysis (KEY_INSIGHT + DEEP_DIVE)
- ✓ Reimbursement Policy Analysis
- ✗ Correlation Analysis (analyzed separately, not fed to recommendations)

### Next Steps

1. Review recommendations for actionability
2. Validate findings with domain experts
3. Prioritize implementation based on estimated impact
4. Monitor outcomes after implementation